In [2]:
import json

# Each row: statement description, clean merchant name, amount, category
transactions = [
    ("VASONA PARK PARKING", "Vasona Park", 8.00, "Entertainment"),
    ("VASONA PARK ENTRY", "Vasona Park", 6.50, "Entertainment"),
    ("TARGET #1284", "Target", 42.18, "Other / Mixed"),
    ("TARGET OPTICAL #1284", "Target Optical", 42.18, "Healthcare"),
]

dataset = []

for description, merchant, amount, category in transactions:
    example = {
        "input": f"10/12 {description} ${amount:.2f}",
        "output": {
            "merchant": merchant,
            "amount": amount,
            "category": category,
        },
    }
    dataset.append(example)

print(f"Generated {len(dataset)} examples")
print(json.dumps(dataset[-1], indent=2))  # Show the last example

def make_prompt(statement):
    return f"""Read this credit card statement line: {statement}

Return one JSON object with three keys: merchant, amount, and category.
Use the merchant name without the store number. Make amount a number.
Choose one category: Groceries, Dining, Shopping, Transportation, Gas,
Entertainment, Healthcare, Travel, Utilities, Subscriptions, or Other / Mixed.
If the merchant is ambiguous, use Other / Mixed. Return only the JSON object."""

print(make_prompt(dataset[-1]["input"]))

Generated 4 examples
{
  "input": "10/12 TARGET OPTICAL #1284 $42.18",
  "output": {
    "merchant": "Target Optical",
    "amount": 42.18,
    "category": "Healthcare"
  }
}
Read this credit card statement line: 10/12 TARGET OPTICAL #1284 $42.18

Return one JSON object with three keys: merchant, amount, and category.
Use the merchant name without the store number. Make amount a number.
Choose one category: Groceries, Dining, Shopping, Transportation, Gas,
Entertainment, Healthcare, Travel, Utilities, Subscriptions, or Other / Mixed.
If the merchant is ambiguous, use Other / Mixed. Return only the JSON object.


In [1]:
%pip -q install "transformers==4.56.2"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "Choose a GPU under Runtime → Change runtime type."

model_id = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, dtype=torch.float16
).to("cuda")
model.eval()

print("Loaded Qwen on", torch.cuda.get_device_name(0))

Loaded Qwen on Tesla T4


In [4]:
statement = dataset[-1]["input"]
messages = [{"role": "user", "content": make_prompt(statement)}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=100, do_sample=False)

# generate() returns the prompt followed by the reply. Read only the reply.
reply_tokens = output[0, inputs["input_ids"].shape[1]:]
prediction = tokenizer.decode(reply_tokens, skip_special_tokens=True)

print("Statement:", statement)
print("Qwen's reply:", prediction)

Statement: 10/12 TARGET OPTICAL #1284 $42.18
Qwen's reply: ```json
{
  "merchant": "TARGET",
  "amount": 4218,
  "category": "Groceries"
}
```


In [9]:

statement = dataset[-1]["input"]
messages = [{"role": "user", "content": make_prompt(statement)}]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    enable_thinking=False,
    return_dict=True,
    return_tensors="pt",
).to("cuda")

with torch.inference_mode():
    output = model.generate(**inputs, max_new_tokens=100, do_sample=False)

# generate() returns the prompt followed by the reply. Read only the reply.
reply_tokens = output[0, inputs["input_ids"].shape[1]:]
prediction = tokenizer.decode(reply_tokens, skip_special_tokens=True)

print("Statement:", statement)
print("Qwen's reply:", prediction)




Statement: 10/12 TARGET OPTICAL #1284 $42.18
Qwen's reply: ```json
{
  "merchant": "TARGET",
  "amount": 4218,
  "category": "Groceries"
}
```


In [10]:
def predict(statement):
    messages = [{"role": "user", "content": make_prompt(statement)}]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=100, do_sample=False)

    reply_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(reply_tokens, skip_special_tokens=True)

In [11]:
import json

baseline_replies = []
valid_count = 0

for example in dataset:
    statement = example["input"]
    reply = predict(statement)
    baseline_replies.append(reply)  # Keep the original text for inspection

    try:
        json.loads(reply)
        valid_count += 1
        status = "Valid JSON"
    except json.JSONDecodeError as error:
        status = f"Invalid JSON: {error.msg}"

    print(f"{statement} → {status}")
    print(reply)
    print()

print(f"Valid JSON: {valid_count}/{len(dataset)}")

10/12 VASONA PARK PARKING $8.00 → Invalid JSON: Expecting value
```json
{
  "merchant": "VASONA PARK",
  "amount": 8,
  "category": "Groceries"
}
```

10/12 VASONA PARK ENTRY $6.50 → Invalid JSON: Expecting value
```json
{
  "merchant": "VASONA PARK",
  "amount": 650,
  "category": "Groceries"
}
```

10/12 TARGET #1284 $42.18 → Invalid JSON: Expecting value
```json
{
  "merchant": "TARGET",
  "amount": 4218,
  "category": "Groceries"
}
```

10/12 TARGET OPTICAL #1284 $42.18 → Invalid JSON: Expecting value
```json
{
  "merchant": "TARGET",
  "amount": 4218,
  "category": "Groceries"
}
```

Valid JSON: 0/4


In [12]:
import json
from google.colab import files

filename = "baseline_dev_4.jsonl"

with open(filename, "w") as f:
    for example, reply in zip(dataset, baseline_replies):
        record = {"input": example["input"], "expected": example["output"], "reply": reply}
        f.write(json.dumps(record) + "\n")

files.download(filename)

In [18]:
import random

rng = random.Random(42)  # Rerunning this cell gives the same examples

# Statement name, expected merchant name, expected category
train_merchants = [
    ("TRADER JOES", "Trader Joe's", "Groceries"),
    ("PRESSED JUICERY", "Pressed Juicery", "Dining"),
    ("LINKEDIN PREMIUM", "LinkedIn Premium", "Subscriptions"),
     ("Trinethra Supermarket", "Trinethra", "Groceries"),

]

train_examples = []

for statement_name, merchant, category in train_merchants:
    for template in ["date_first", "amount_first"]:
        date = f"10/{rng.randint(1, 31):02d}"
        cents = rng.randint(500, 7500)
        amount = cents / 100

        if template == "date_first":
            statement = f"{date} {statement_name} ${amount:.2f}"
        else:
            statement = f"${amount:.2f} | {statement_name} | {date}"

        train_examples.append({
            "input": statement,
            "output": {"merchant": merchant, "amount": amount, "category": category},
        })

for example in train_examples:
    print(example["input"], "→", example["output"])

10/21 TRADER JOES $14.12 → {'merchant': "Trader Joe's", 'amount': 14.12, 'category': 'Groceries'}
$65.74 | TRADER JOES | 10/01 → {'merchant': "Trader Joe's", 'amount': 65.74, 'category': 'Groceries'}
10/09 PRESSED JUICERY $25.06 → {'merchant': 'Pressed Juicery', 'amount': 25.06, 'category': 'Dining'}
$16.43 | PRESSED JUICERY | 10/08 → {'merchant': 'Pressed Juicery', 'amount': 16.43, 'category': 'Dining'}
10/24 LINKEDIN PREMIUM $13.39 → {'merchant': 'LinkedIn Premium', 'amount': 13.39, 'category': 'Subscriptions'}
$65.67 | LINKEDIN PREMIUM | 10/22 → {'merchant': 'LinkedIn Premium', 'amount': 65.67, 'category': 'Subscriptions'}
10/29 Trinethra Supermarket $49.67 → {'merchant': 'Trinethra', 'amount': 49.67, 'category': 'Groceries'}
$53.37 | Trinethra Supermarket | 10/03 → {'merchant': 'Trinethra', 'amount': 53.37, 'category': 'Groceries'}


In [19]:
merchants_by_split = {
    "train": train_merchants,  # Reuse the list you already built
    "validation": [
        ("INDIA CASH N CURRY", "India Cash N Curry", "Groceries"),
        ("CHAAT BhAVAN", "Chaat Bhavan", "Dining"),
    ],
    "test": [
        ("BHARAT BAZAAR", "Bharat Bazaar", "Groceries"),
         ("WILLOW BISTRO", "Willow Bistro", "Dining")
    ],
}

formats_by_split = {
    "train": ["{date} {name} ${amount:.2f}", "${amount:.2f} | {name} | {date}"],
    "validation": ["POS {name} USD {amount:.2f} ON {date}"],
    "test": ["{name} / PURCHASE {date} / ${amount:.2f}"],
}

for split, merchants in merchants_by_split.items():
    print(split, "→", [merchant for _, merchant, _ in merchants])

train → ["Trader Joe's", 'Pressed Juicery', 'LinkedIn Premium', 'Trinethra']
validation → ['India Cash N Curry', 'Chaat Bhavan']
test → ['Bharat Bazaar', 'Willow Bistro']


In [20]:
import random

def generate_split(merchants, formats, seed):
    rng = random.Random(seed)
    examples = []

    for statement_name, merchant, category in merchants:
        for template in formats:
            date = f"10/{rng.randint(1, 31):02d}"
            amount = rng.randint(500, 7500) / 100

            statement = template.format(date=date, name=statement_name, amount=amount)
            examples.append({
                "input": statement,
                "output": {"merchant": merchant, "amount": amount, "category": category},
            })

    return examples

splits = {}
for split, seed in [("train", 42), ("validation", 43), ("test", 44)]:
    splits[split] = generate_split(
        merchants_by_split[split], formats_by_split[split], seed
    )
    print(split, len(splits[split]))

print(splits["train"][0])

train 8
validation 2
test 2
{'input': '10/21 TRADER JOES $14.12', 'output': {'merchant': "Trader Joe's", 'amount': 14.12, 'category': 'Groceries'}}


In [21]:
new_categories = {
    "Shopping":       ("Macys", "Nordstorm Rack", "DSW"),
    "Transportation": ("Caltrain", "VTA", "Uber"),
    "Gas":            ("Chevron", "Snell Gas Station", "7/11"),
    "Entertainment":  ("Cinemark", "AMC", "Harbor Aquarium"),
    "Healthcare":     ("Oak Dental", "Clearview Optometry", "Valley Clinic"),
    "Travel":         ("United Airlines", "Alaska Airlines", "Airbnb"),
    "Utilities":      ("City Water", "PG&E", "Tesla"),
    "Other / Mixed":  ("Costco Wholesale", "Amazon Marketplace", "Walmart Supercenter"),
}

# Copy your existing lists so this cell is safe to rerun.
catalog = {split: list(rows) for split, rows in merchants_by_split.items()}

for category, merchants in new_categories.items():
    for split, merchant in zip(("train", "validation", "test"), merchants):
        catalog[split].append((merchant.upper(), merchant, category))

# LinkedIn Premium is already the training example for Subscriptions.
catalog["validation"].append(("WHOOP MEMBERSHIP", "WHOOP", "Subscriptions"))
catalog["test"].append(("SUBSTACK SUBSCRIPTION", "Substack", "Subscriptions"))

In [22]:
for split, seed in [("train", 42), ("validation", 43), ("test", 44)]:
    splits[split] = generate_split(catalog[split], formats_by_split[split], seed)
    categories = {row["output"]["category"] for row in splits[split]}
    print(f"{split}: {len(splits[split])} examples, {len(categories)} categories")

train: 24 examples, 11 categories
validation: 11 examples, 11 categories
test: 11 examples, 11 categories


The next step is to generate the 396 rows once, check them, and download a frozen copy. I prepared the [merchant catalog](https://github.com/Shrezz/credit-card-llm-finetuning/blob/main/data/merchant_catalog_v1.json) so you don’t have to type 264 statement aliases by hand. I checked it with the generator you’ve been using: it produces 264 training, 66 validation, and 66 test examples, with all 11 categories balanced within each split. No merchant crosses splits, and your four Target and Vasona Park demo examples stay separate.

1. Download the [catalog](https://github.com/Shrezz/credit-card-llm-finetuning/blob/main/data/merchant_catalog_v1.json), then run this cell in Colab. Choose that file when prompted. This reuses your existing generate_split() and formats_by_split; if you restarted Colab, rerun the cells that define them first.

In [24]:
import json
from collections import Counter
from google.colab import files

uploaded = files.upload()
catalog = json.loads(next(iter(uploaded.values())))

splits = {}
for split, seed in [("train", 42), ("validation", 43), ("test", 44)]:
    rows = [
        (alias, m["merchant"], m["category"])
        for m in catalog[split]
        for alias in m["aliases"]
    ]
    splits[split] = generate_split(rows, formats_by_split[split], seed)

    counts = Counter(row["output"]["category"] for row in splits[split])
    per_category = 24 if split == "train" else 6
    assert len(counts) == 11 and set(counts.values()) == {per_category}
    print(f"{split}: {len(splits[split])} rows, {per_category} per category")

Saving merchant_catalog_v1.json to merchant_catalog_v1 (1).json
train: 264 rows, 24 per category
validation: 66 rows, 6 per category
test: 66 rows, 6 per category


In [25]:
import hashlib
from zipfile import ZipFile

archive_name = "transactions_v1.zip"

with ZipFile(archive_name, "x") as archive:  # "x" prevents overwriting an existing copy
    for split, examples in splits.items():
        data = "".join(json.dumps(row) + "\n" for row in examples).encode()
        archive.writestr(f"{split}.jsonl", data)

        if split == "test":
            fingerprint = hashlib.sha256(data).hexdigest()
            archive.writestr("test.sha256", fingerprint + "\n")
            print("Test fingerprint:", fingerprint)

files.download(archive_name)

Test fingerprint: 4a4b94dba23f908ddb5108f9806b3ec58302144406ab538f72aceb9ff97963ae


Cell 1 — read the frozen test set. If you’re in the same Colab session, the ZIP is still there. If you restarted, upload transactions_v1.zip to Colab’s Files panel first.


Cell 2 — run the baseline and download the predictions. Use the original Qwen model and the same predict() function that produced your four demo replies. If you restarted Colab, rerun your make_prompt(), model-loading, and predict() cells first.

In [26]:
import hashlib
import json
from zipfile import ZipFile

with ZipFile("transactions_v1.zip") as archive:
    test_bytes = archive.read("test.jsonl")

test_sha = hashlib.sha256(test_bytes).hexdigest()
assert test_sha == "4a4b94dba23f908ddb5108f9806b3ec58302144406ab538f72aceb9ff97963ae"

test_rows = [json.loads(line) for line in test_bytes.splitlines()]
assert len(test_rows) == 66
print("Verified: 66 frozen test examples")

Verified: 66 frozen test examples


In [27]:
from google.colab import files

filename = "qwen3_baseline_test66.jsonl"

with open(filename, "w") as f:
    for i, row in enumerate(test_rows, start=1):
        reply = predict(row["input"])  # Qwen sees only the statement

        record = {
            "id": i,
            "input": row["input"],
            "expected": row["output"],
            "reply": reply,
        }
        f.write(json.dumps(record) + "\n")

        if i % 10 == 0 or i == len(test_rows):
            print(f"Saved {i}/{len(test_rows)} predictions")

files.download(filename)

Saved 10/66 predictions
Saved 20/66 predictions
Saved 30/66 predictions
Saved 40/66 predictions
Saved 50/66 predictions
Saved 60/66 predictions
Saved 66/66 predictions


Next, score the 66 saved predictions. You don’t need to run Qwen again. This step is ordinary Python: compare each raw reply with its expected answer.

We’ll use the same grading rules before and after fine-tuning:
- JSON validity: The whole reply must parse as JSON. Markdown fences count as a failure.
- Merchant: Ignore capitalization and surrounding spaces, so TARGET matches Target, but not Target Optical.
- Amount: Require a JSON number with the correct value. 8 matches 8.00; 650 does not match 6.50.
- Category: Require the exact label, such as Other / Mixed.
All four scores use 66 as the denominator. If a reply can’t be parsed, it earns no field credit. That measures whether a program could actually use the answer as returned.

In [28]:
import json
from decimal import Decimal
from google.colab import files

def reject_constant(value):
    raise ValueError(f"{value} is not a valid JSON number")

def score_predictions(filename):
    with open(filename) as f:
        rows = [json.loads(line, parse_float=Decimal) for line in f]

    assert len(rows) == 66, f"Expected 66 predictions, found {len(rows)}"
    correct = {"json_validity": 0, "merchant": 0, "amount": 0, "category": 0}

    for row in rows:
        gold = row["expected"]
        try:
            pred = json.loads(row["reply"], parse_float=Decimal,
                              parse_constant=reject_constant)
        except ValueError:
            continue

        correct["json_validity"] += 1
        if not isinstance(pred, dict):
            continue  # Valid JSON, but no named fields to grade

        merchant = pred.get("merchant")
        if isinstance(merchant, str):
            correct["merchant"] += merchant.strip().casefold() == gold["merchant"].strip().casefold()

        amount = pred.get("amount")
        if type(amount) in (int, Decimal):
            correct["amount"] += amount == gold["amount"]

        correct["category"] += pred.get("category") == gold["category"]

    return {
        metric: {"correct": n, "total": len(rows), "percent": round(100 * n / len(rows), 1)}
        for metric, n in correct.items()
    }

metrics = score_predictions("qwen3_baseline_test66.jsonl")
for metric, result in metrics.items():
    print(f"{metric}: {result['correct']}/{result['total']} ({result['percent']}%)")

with open("baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
files.download("baseline_metrics.json")

json_validity: 0/66 (0.0%)
merchant: 0/66 (0.0%)
amount: 0/66 (0.0%)
category: 0/66 (0.0%)


QuickCheck before preparing traning pairs.
We know all 66 raw replies failed to parse. The useful question is narrower: how many become readable JSON objects if we remove only the outer Markdown fences? That tells you what caused those failures without changing the baseline score.

In [29]:
import json

with open("qwen3_baseline_test66.jsonl") as f:
    records = [json.loads(line) for line in f]

recovered = 0
for record in records:
    lines = record["reply"].strip().splitlines()
    if len(lines) < 3:
        continue
    if lines[0].strip().lower() not in ("```json", "```") or lines[-1].strip() != "```":
        continue

    inside = "\n".join(lines[1:-1])
    try:
        obj = json.loads(inside, parse_constant=reject_constant)
    except ValueError:
        continue

    if isinstance(obj, dict):
        recovered += 1

print(f"Readable JSON objects after removing only fences: {recovered}/{len(records)}")

Readable JSON objects after removing only fences: 66/66


Your baseline is complete: 0 of 66 raw replies parsed as JSON. Under your scoring rules, that makes merchant, amount, and category accuracy 0/66 too. That doesn’t mean every value was wrong. In your earlier Vasona Park example, Qwen returned the correct amount (8), but the Markdown fence prevented the parser from reading it. We know fences affected the four demo replies; we haven’t established why each of the 66 test replies failed.

Keep baseline_metrics.json and the raw predictions unchanged. Next, prepare the examples that Qwen will learn from. Each training example has two parts: the same prompt you used for the baseline, and the answer you want Qwen to generate. That answer should be plain JSON.

In [30]:
import json
from zipfile import ZipFile

with ZipFile("transactions_v1.zip") as archive:
    train_rows = [
        json.loads(line)
        for line in archive.read("train.jsonl").splitlines()
    ]

def make_training_pair(row):
    return {
        "prompt": [{"role": "user", "content": make_prompt(row["input"])}],
        "completion": [{"role": "assistant", "content": json.dumps(row["output"])}],
    }

train_pairs = [make_training_pair(row) for row in train_rows]

print("Training pairs:", len(train_pairs))
print("\nPROMPT:\n", train_pairs[0]["prompt"][0]["content"])
print("\nEXPECTED REPLY:\n", train_pairs[0]["completion"][0]["content"])

Training pairs: 264

PROMPT:
 Read this credit card statement line: 10/21 TRADER JOES $14.12

Return one JSON object with three keys: merchant, amount, and category.
Use the merchant name without the store number. Make amount a number.
Choose one category: Groceries, Dining, Shopping, Transportation, Gas,
Entertainment, Healthcare, Travel, Utilities, Subscriptions, or Other / Mixed.
If the merchant is ambiguous, use Other / Mixed. Return only the JSON object.

EXPECTED REPLY:
 {"merchant": "Trader Joe's", "amount": 14.12, "category": "Groceries"}


Cell 1 — install the training libraries. This keeps the same Transformers version you used for the baseline and satisfies TRL’s version requirements.

In [31]:
%pip -q install "transformers==4.56.2" "peft==0.17.1" "trl==0.23.1"

Cell 2 — attach the adapter once to the original Qwen model:

In [32]:
from transformers import set_seed
from peft import LoraConfig, get_peft_model

set_seed(42)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,146,880 || all params: 597,196,800 || trainable%: 0.1920


Cell 1 — connect the training and validation examples. We’ll use the same Qwen chat template and thinking setting as your baseline. The 66 validation rows come from validation.jsonl; your 66 test rows stay set aside.

In [33]:
import json
from zipfile import ZipFile
from datasets import Dataset

with ZipFile("transactions_v1.zip") as archive:
    val_rows = [json.loads(line) for line in archive.read("validation.jsonl").splitlines()]

def trainer_example(pair):
    return {
        "prompt": tokenizer.apply_chat_template(
            pair["prompt"],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        ),
        "completion": pair["completion"][0]["content"],
    }

train_ds = Dataset.from_list([trainer_example(p) for p in train_pairs])
val_ds = Dataset.from_list([trainer_example(make_training_pair(r)) for r in val_rows])

print(f"Training: {len(train_ds)} | Validation: {len(val_ds)}")

Training: 264 | Validation: 66


Cell 2 — set up the trainer and measure the starting loss. completion_only_loss=True tells TRL’s trainer to score the answer tokens rather than the instructions you supplied.

In [34]:
from trl import SFTConfig, SFTTrainer

args = SFTConfig(
    output_dir="qwen_txn_lora",
    num_train_epochs=1,
    learning_rate=1e-4,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,
    max_length=256,
    completion_only_loss=True,
    fp16=True,
    bf16=False,
    gradient_checkpointing=False,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,                 # Already has your LoRA adapter
    args=args,
    processing_class=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
)

starting_loss = trainer.evaluate()["eval_loss"]
print(f"Validation loss before training: {starting_loss:.4f}")

Validation loss before training: 0.8406


Now you’re ready to train.

In [35]:
training_run = trainer.train()

# Your trainer evaluates automatically at the end of the epoch.
ending_loss = next(
    entry["eval_loss"]
    for entry in reversed(trainer.state.log_history)
    if "eval_loss" in entry
)

print(f"Validation loss before training: {starting_loss:.4f}")
print(f"Validation loss after training:  {ending_loss:.4f}")

Validation loss before training: 0.8406
Validation loss after training:  0.1912


In [36]:
import json
import shutil
from pathlib import Path
from google.colab import files

adapter_dir = Path("transaction_lora_adapter")
trainer.model.save_pretrained(adapter_dir)

with (adapter_dir / "training_log.json").open("w") as f:
    json.dump({
        "starting_validation_loss": starting_loss,
        "ending_validation_loss": ending_loss,
        "history": trainer.state.log_history,
    }, f, indent=2)

files.download(shutil.make_archive(str(adapter_dir), "zip", root_dir=adapter_dir))

Your validation loss fell from 0.8406 to 0.1912 after one epoch. That means the adapter got better at predicting the expected answer tokens on the validation examples. Now you can test the more practical question: when Qwen writes the whole answer itself, does it produce valid JSON with the right values?

Cell 1 — generate the fine-tuned predictions. Your existing predict() function uses the variable model. Point it at the trained adapter, keep the original prompt and decoding settings, and read the same frozen test file.

In [37]:
import hashlib
import json
from zipfile import ZipFile

model = trainer.model
model.eval()  # Turns off training behavior such as dropout; the adapter stays active

with ZipFile("transactions_v1.zip") as archive:
    test_bytes = archive.read("test.jsonl")

assert hashlib.sha256(test_bytes).hexdigest() == (
    "4a4b94dba23f908ddb5108f9806b3ec58302144406ab538f72aceb9ff97963ae"
)
test_rows = [json.loads(line) for line in test_bytes.splitlines()]
assert len(test_rows) == 66

ft_file = "qwen3_finetuned_test66.jsonl"
with open(ft_file, "w") as f:
    for i, row in enumerate(test_rows, start=1):
        reply = predict(row["input"])  # Only the statement goes to Qwen
        record = {
            "id": i,
            "input": row["input"],
            "expected": row["output"],
            "reply": reply,
        }
        f.write(json.dumps(record) + "\n")
        if i % 10 == 0 or i == 66:
            print(f"Saved {i}/66 predictions")

Saved 10/66 predictions
Saved 20/66 predictions
Saved 30/66 predictions
Saved 40/66 predictions
Saved 50/66 predictions
Saved 60/66 predictions
Saved 66/66 predictions


Cell 2 — grade both runs by the same rules. This reuses your original score_predictions() function, so Markdown fences still count as invalid and every metric keeps 66 as its denominator. It prints the comparison and downloads the results together.

In [38]:
from pathlib import Path
from google.colab import files

base = json.loads(Path("baseline_metrics.json").read_text())
tuned = score_predictions(ft_file)
Path("finetuned_metrics.json").write_text(json.dumps(tuned, indent=2))

lines = [
    "| Metric | Base model | Fine-tuned | Change |",
    "|---|---:|---:|---:|",
]
for key, label in [
    ("category", "Category accuracy"),
    ("merchant", "Merchant accuracy"),
    ("amount", "Amount accuracy"),
    ("json_validity", "JSON validity"),
]:
    b, t = base[key], tuned[key]
    assert b["total"] == t["total"] == 66
    change = 100 * (t["correct"] - b["correct"]) / 66
    lines.append(
        f"| {label} | {b['correct']}/66 ({b['percent']:.1f}%) "
        f"| {t['correct']}/66 ({t['percent']:.1f}%) | {change:+.1f} pp |"
    )

comparison = "\n".join(lines)
print(comparison)
Path("comparison.md").write_text(comparison + "\n")

with ZipFile("evaluation_results.zip", "w") as archive:
    for name in [
        "qwen3_baseline_test66.jsonl", ft_file,
        "baseline_metrics.json", "finetuned_metrics.json", "comparison.md",
    ]:
        archive.write(name)

files.download("evaluation_results.zip")

| Metric | Base model | Fine-tuned | Change |
|---|---:|---:|---:|
| Category accuracy | 0/66 (0.0%) | 50/66 (75.8%) | +75.8 pp |
| Merchant accuracy | 0/66 (0.0%) | 42/66 (63.6%) | +63.6 pp |
| Amount accuracy | 0/66 (0.0%) | 66/66 (100.0%) | +100.0 pp |
| JSON validity | 0/66 (0.0%) | 66/66 (100.0%) | +100.0 pp |


Fine-tuning improved all four measured scores on your frozen test set. After one epoch, Qwen produced usable JSON and extracted the correct amount on all 66 transactions. It chose the correct category on 50/66 and returned the expected merchant name on 42/66. Merchant extraction was the weakest field by your scoring rule.

The clearest finding is the change from 0/66 to 66/66 directly parseable replies. Be precise about the baseline in your write-up: its field scores were zero because none of its raw replies parsed. That doesn’t mean every value inside those replies was wrong. The comparison measures whether a program could read the answer and use the field as returned.

Next, find out how often the entire transaction was correct. You can’t tell that from the table alone: some of the 16 category mistakes may overlap with the 24 merchant mistakes. Since every fine-tuned reply parsed and every amount matched, you only need to check merchant and category together. This runs on the saved file without using Qwen again:

In [39]:
import json

fully_correct = 0
examples_shown = 0

with open("qwen3_finetuned_test66.jsonl") as f:
    for line in f:
        row = json.loads(line)
        pred = json.loads(row["reply"])
        gold = row["expected"]

        merchant = pred.get("merchant")
        merchant_ok = (
            isinstance(merchant, str)
            and merchant.strip().casefold() == gold["merchant"].strip().casefold()
        )
        category_ok = pred.get("category") == gold["category"]

        if merchant_ok and category_ok:
            fully_correct += 1
        elif examples_shown < 3:
            print("Statement:", row["input"])
            print("Expected: ", gold)
            print("Predicted:", pred, "\n")
            examples_shown += 1

print(f"All three fields correct: {fully_correct}/66")

Statement: LAKEVIEW MKT / PURCHASE 10/08 / $28.72
Expected:  {'merchant': 'Lakeview Market', 'amount': 28.72, 'category': 'Groceries'}
Predicted: {'merchant': 'Lakeview Mkt', 'amount': 28.72, 'category': 'Other / Mixed'} 

Statement: LAKEVIEW MARKET #12 / PURCHASE 10/01 / $23.40
Expected:  {'merchant': 'Lakeview Market', 'amount': 23.4, 'category': 'Groceries'}
Predicted: {'merchant': 'Lakeview Market', 'amount': 23.4, 'category': 'Shopping'} 

Statement: WILLOW BISTRO / PURCHASE 10/04 / $51.58
Expected:  {'merchant': 'Willow Bistro', 'amount': 51.58, 'category': 'Dining'}
Predicted: {'merchant': 'Willy Bistro', 'amount': 51.58, 'category': 'Groceries'} 

All three fields correct: 38/66
